# 01 · Where the Money Goes

**Module 09 · Cost Optimization**

A cent per email sounds like nothing. But the desk answers emails all day, and **a cent, repeated, is a bill**. This module takes the same support desk and makes it cheaper **without making it worse**.

There is one rule for the whole module:

> **No optimisation without a measurement.** Every trick must show, on the same emails, what it saved **and** whether the answers stayed correct.

| Notebook | What you will learn |
|---|---|
| 1 | **Where the money goes** (this one): how a call is priced, and a first look at cheap vs strong |
| 2 | **The right model for the job**: routing easy emails to a cheap model |
| 3 | **Don't pay twice**: prompt caching, an exact cache, and a semantic cache (and its danger) |
| 4 | **Don't pay full price**: sending less, and the Batch API |
| 5 | **Put it together**: five setups, one table, and a monthly bill |

> This module makes small OpenAI calls: about 50 cents for all five notebooks. Some cells take a minute.

In [1]:
import inbox
import llm

print("Ready. Models:", llm.SMALL, "(cheap) and", llm.BIG, "(strong)")

Ready. Models: gpt-4o-mini (cheap) and gpt-4o (strong)


## 1. How one call is priced
You pay **per token** (a token is about three quarters of a word). There are three prices, in dollars per **million** tokens:

| | Input (what you send) | Cached input | Output (what the model writes) |
|---|---|---|---|
| `gpt-4o-mini` | $0.15 | $0.075 | $0.60 |
| `gpt-4o` | $2.50 | $1.25 | $10.00 |

(Prices change. They live in one table, `PRICES` in `llm.py`. Check the OpenAI pricing page if you run this much later.)

Two things stand out: **output costs 4 times more than input**, and **`gpt-4o` costs about 16 times more than `gpt-4o-mini`**. Here is how we compute a call's cost:

*From `llm.py`:*

```python
def cost_of(model, prompt_tokens, cached_tokens, completion_tokens, batch=False):
    """What did one call cost, in dollars?
    Cached input tokens are cheaper than normal ones. A batch call costs half."""
    price = price_for(model)
    normal_tokens = prompt_tokens - cached_tokens
    dollars = (normal_tokens * price[0] + cached_tokens * price[1] + completion_tokens * price[2]) / 1_000_000
    if batch:
        dollars = dollars * BATCH_DISCOUNT
    return dollars
```

Every call in this module goes through `llm.ask`, which returns the reply **together with** its tokens and dollars:

*From `llm.py`:*

```python
def ask(user_text, model=SMALL, system_text=None, max_tokens=150):
    """Ask the model one question. Returns the reply together with what the call cost.

    The system text (the handbook) goes FIRST and never changes. The customer's email goes LAST.
    That order matters: OpenAI reuses (caches) a long prompt start that it has seen before. See notebook 03."""
    if system_text is None:
        system_text = HANDBOOK

    started = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text},
        ],
        max_tokens=max_tokens,   # a ceiling on the reply length: you pay for every output token
        temperature=0,
    )
    seconds = time.time() - started

    usage = response.usage
    cached = 0
    if usage.prompt_tokens_details is not None and usage.prompt_tokens_details.cached_tokens is not None:
        cached = usage.prompt_tokens_details.cached_tokens

    return {
        "text": response.choices[0].message.content,
        "model": model,
        "prompt_tokens": usage.prompt_tokens,
        "cached_tokens": cached,
        "completion_tokens": usage.completion_tokens,
        "cost": cost_of(model, usage.prompt_tokens, cached, usage.completion_tokens),
        "seconds": seconds,
    }
```

In [2]:
result = llm.ask("How much does express shipping cost?", llm.SMALL)

print("Reply            :", result["text"])
print()
print("Input tokens     :", result["prompt_tokens"], "  (of which cached:", result["cached_tokens"], ")")
print("Output tokens    :", result["completion_tokens"])
print("Cost of the call : $" + format(result["cost"], ".6f"))

Reply            : Express shipping always costs $12. If you have any other questions, a colleague can help.

Input tokens     : 1154   (of which cached: 1024 )
Output tokens    : 19
Cost of the call : $0.000108


## 2. Where do the tokens come from?
The customer's question is five words. But we sent over a thousand input tokens. The rest is the **handbook**: the shop's rules, which the model needs to answer correctly. Real system prompts are long like this.

In [3]:
price = llm.price_for(llm.SMALL)
input_part = (result["prompt_tokens"] - result["cached_tokens"]) * price[0] / 1_000_000
cached_part = result["cached_tokens"] * price[1] / 1_000_000
output_part = result["completion_tokens"] * price[2] / 1_000_000

print("The cost of this one call, by part:")
print("  input  (the handbook + the question):  $" + format(input_part, ".6f"))
print("  cached input                        :  $" + format(cached_part, ".6f"))
print("  output (the reply)                  :  $" + format(output_part, ".6f"))
print()
print("The handbook alone is", len(llm.HANDBOOK.split()), "words.")

The cost of this one call, by part:
  input  (the handbook + the question):  $0.000019
  cached input                        :  $0.000077
  output (the reply)                  :  $0.000011

The handbook alone is 920 words.


**For this desk, almost all of the money is spent on input, and most of that input is the same handbook, sent again and again.** That already tells us where to look: *send it less* (notebook 4) or *get it cheaper* (caching, notebook 3).

## 3. The same question, two models

In [4]:
question = "My cart is $62. Will I pay anything for standard shipping?"

for model in [llm.SMALL, llm.BIG]:
    answer = llm.ask(question, model)
    print(model.ljust(12), "$" + format(answer["cost"], ".6f"), " ", answer["text"])

print()
print("Per 1,000 emails the difference is about $" + format(llm.ask(question, llm.BIG)["cost"] * 1000, ".2f")
      + " against $" + format(llm.ask(question, llm.SMALL)["cost"] * 1000, ".2f"))

gpt-4o-mini  $0.000109   No, standard shipping is free on orders over $50. You won't pay anything for shipping.


gpt-4o       $0.001742   No, standard shipping is free on orders over $50.



Per 1,000 emails the difference is about $1.74 against $0.11


Both answered correctly. One cost about 16 times more. So **do we ever need the strong model?** We cannot answer that by guessing. We need to *test*.

## 4. What does "correct" mean? A test set
We cannot make anything cheaper without knowing it is still **good**. So every email in `inbox.py` has a list of **facts** that a correct reply must contain, taken from the handbook. The score is the share of facts that appear:

*From `inbox.py`:*

```python
def score_reply(reply, facts):
    """The share of required facts that appear in the reply (0.0 to 1.0)."""
    text = reply.lower()
    found = 0
    for alternatives in facts:
        for spelling in alternatives:
            if spelling.lower() in text:
                found = found + 1
                break
    return found / len(facts)
```

In [5]:
print("Emails in the test set:", len(inbox.EMAILS))
easy = 0
for item in inbox.EMAILS:
    if item["kind"] == "easy":
        easy = easy + 1
print("  easy (one simple question):", easy)
print("  hard (several questions, arithmetic, or a person must take over):", len(inbox.EMAILS) - easy)

item = inbox.EMAILS[12]
print()
print("Example (hard):", item["email"])
print("Facts a correct reply must contain:", item["facts"])

Emails in the test set: 28
  easy (one simple question): 12
  hard (several questions, arithmetic, or a person must take over): 16

Example (hard): I bought headphones 25 days ago and the left side stopped working. Can I get a refund, who pays for return shipping, and how long until the money arrives?
Facts a correct reply must contain: [['30 day', 'within'], ['free', 'we pay', 'no cost'], ['5 to 7', '5-7']]


The 12 easy emails are one-line questions. The 16 hard ones ask several things at once, need arithmetic that combines two rules, or (like a lawyer threat or a card number) must be **handed to a person**.

> **A confession about this test set.** We first wrote 24 emails. The cheap model passed **all** of them, which means they could not tell us anything. A test set is only useful once it contains something that can **fail**. So we added four emails with arithmetic that combines two rules. That is exactly how real test sets grow: from failures.

## 5. A first look: where does the cheap model fail?
The four arithmetic emails, to both models:

In [6]:
arithmetic = inbox.EMAILS[24:]

for model in [llm.SMALL, llm.BIG]:
    print("=====", model)
    for item in arithmetic:
        answer = llm.ask(item["email"], model, max_tokens=200)
        score = inbox.score_reply(answer["text"], item["facts"])
        print("score", format(score, ".1f"), "|", item["email"][:62])
        if score < 1.0:
            print("        reply:", answer["text"][:200].replace("\n", " "))

===== gpt-4o-mini


score 0.5 | I ordered three items: $19.99, $14.50 and $9.00. Will I pay fo
        reply: Yes, you will get free standard shipping since your order total is over $50. For the opened, not faulty $14.50 item, a 10% restocking fee applies, which is $1.45. You will also pay for the return ship


score 0.0 | My order is $49.50. How much more would express shipping cost 
        reply: Express shipping costs $12, while standard shipping is free for orders over $50. You would need to spend an additional $0.50 to qualify for free standard shipping, making express shipping $12 more exp


score 1.0 | I have two orders: a $30 order and a $55 order, both with stan


score 1.0 | I paid $120 for shoes, delivered 20 days ago, opened and not f
===== gpt-4o


score 1.0 | I ordered three items: $19.99, $14.50 and $9.00. Will I pay fo


score 1.0 | My order is $49.50. How much more would express shipping cost 


score 1.0 | I have two orders: a $30 order and a $55 order, both with stan


score 1.0 | I paid $120 for shoes, delivered 20 days ago, opened and not f


The strong model does the arithmetic; the cheap model sometimes slips (for example, it may say shipping is free when the order is under $50). **So the cheap model is not enough for everything, and the strong model is wasteful for everything.** The next notebook finds a middle way.

## What you learned
- You pay per token: input, cached input and output are different prices, and the strong model costs about 16 times more.
- On this desk the **input** (the handbook) is where the tokens go.
- **Measure quality with a test set first.** Checking for required facts is cheap and repeatable.
- A test set must contain things that can fail.

**Next:** use the strong model only where it is needed.